In [0]:
%pip install sentence-transformers

In [0]:
jobs_df = spark.table(
    "career_os.silver.jobs_clean"
)

display(
    jobs_df.select(
        "job_id",
        "title",
        "company",
        "search_text"
    )
)

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [0]:
jobs = jobs_df.select(
    "job_id",
    "search_text"
).collect()

In [0]:
embeddings = []

for row in jobs:
    vector = model.encode(
        row.search_text
    ).tolist()

    embeddings.append(
        (
            row.job_id,
            row.search_text,
            vector
        )
    )

In [0]:
from pyspark.sql.types import *

embedding_schema = StructType([
    StructField(
        "job_id",
        StringType()
    ),
    StructField(
        "search_text",
        StringType()
    ),
    StructField(
        "embedding",
        ArrayType(FloatType())
    )
])


embedding_df = spark.createDataFrame(
    embeddings,
    schema=embedding_schema
)

display(embedding_df)

In [0]:
embedding_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "career_os.ai.job_embeddings"
    )